# Logistic regression

## What is classification?

Classification deals with the prediction of categorical variables, usually with a focus on [nominal variables](1:data_types:variable_types). Logistic regression is the simplest approach for classification. It can be seen as an extension of linear regression to handle categorical variables by predicting the probability of obtaining of given category through a non-linear model.

### Binary logistic regression

Binary logistic regression deals with predicting two categories and, with a single predictor, takes the form ({numref}`figure {number} <logistic_model_params>`):

$$P(Y_i = 1) = \frac{1}{1 + e^{-\left(\beta_0 + \beta_1 x_i\right)}} + \epsilon_i$$

Where:
  * $Y_i$ is the outcome.
  * $x_i$ is an observation of the predictor.
  * $\beta_0$ is the intercept.
  * $\beta_1$ is the slope.
  * $\epsilon_i$ is the error.

````{iframe-figure} ../../_static/part-b_logistic-model_params.html
:name: logistic_model_params
:width: 645px
:height: 400px

Change the slope and origin to see the impact on the model.
````

At the heart of logisitic regression is the logistic function, which is a sigmoid function converting an input between $-\infty$ and $\infty$ into an output between 0 and 1:

$$\text{logistic}(x) = p = \frac{1}{1 + e^{-x}} = \frac{1}{1 + e^{-x}}\cdot\frac{e^x}{e^x} = \frac{e^x}{1 + e^x}$$

Applying a linear regression to predict $P(Y_i = 1)$ leads to an output between $-\infty$ and $\infty$, which would not be a valid probability, hence the use of the logistic function. But it is possible to look at logistic regression from a linear perspective. For that, we need the inverse of the logistic function, called the logit function:

$$\text{logit}(p) = \text{logistic}^{-1}(p) = \log\left(\frac{p}{1 - p}\right)$$

The logit is linked to the concept of odds, which is another way of quantifying uncertainty. While a probability is the ratio of the number of outcomes where an event happens to the total number of outcomes, odds are the ratio of the number of outcomes where an event happens to the number of outcomes where the event does not happen. To put it another way, odds are the ratio of the probability that an event will happen to the probability that it will not. So, in the binary case:

$$O(Y_i = 1) = \frac{P(Y_i = 1)}{1 - P(Y_i = 1)}$$

So the logit is also called the log-odds, which behaves linearly:

$$\log\left(\frac{P(Y_i = 1)}{1 - P(Y_i = 1)}\right) = \beta_0 + \beta_1 x_i$$

Just like the linear model, logistic regression can be reframed as a probabilistic model, but based on a [Bernoulli distribution](1:DiscreteRandom:Bernoulli):

$$y_i \sim \text{Bernoulli}\left(\frac{1}{1 + e^{-\left(\beta_0 + \beta_1 x_i\right)}}\right)$$

And you can also express it in matrix form, making it easier to introduce more than one predictor:

$$P(Y_i = 1) = \frac{1}{1 + e^{-\boldsymbol{\beta X_i}}} = \frac{e^{\boldsymbol{\beta X_i}}}{1 + e^{\boldsymbol{\beta X_i}}}$$

Where:
  * $\boldsymbol{X_i}$ is a vector of predictors for observation $i$, with an extra element with value 1 added for the intercept.
  * $\boldsymbol{\beta}$ is a vector of parameters, i.e., the intercept and a slope for each predictor.

### Multinomial logistic regression

In binary logistic regression, we only model one of the two categories explicitly. Predicting both categories explicitly requires:

$$
    \begin{align}
        P(Y_i = 1) &= \frac{e^{\boldsymbol{\beta_1 X_i}}}{e^{\boldsymbol{\beta_0 X_i}} + e^{\boldsymbol{\beta_1 X_i}}} \\
        P(Y_i = 0) &= \frac{e^{\boldsymbol{\beta_0 X_i}}}{e^{\boldsymbol{\beta_0 X_i}} + e^{\boldsymbol{\beta_1 X_i}}} \\
    \end{align}
$$

In that setting, you have two sets of parameters: one for category 1 and another for category 0. But now the model is non-identifiable: multiple values of $\boldsymbol{\beta_0}$ and $\boldsymbol{\beta_1}$ can lead to the same outcomes. You can see that by adding a constant $\boldsymbol{C}$ to those parameters:

$$
    \begin{align}
        P(Y_i = 1) &= \frac{e^{(\boldsymbol{\beta_1} + \boldsymbol{C})\boldsymbol{X_i}}}{e^{(\boldsymbol{\beta_0} + \boldsymbol{C})\boldsymbol{X_i}} + e^{(\boldsymbol{\beta_1} + \boldsymbol{C})\boldsymbol{X_i}}} \\
         &= \frac{e^{\boldsymbol{C X_i}}e^{\boldsymbol{\beta_1 X_i}}}{e^{\boldsymbol{C X_i}}e^{\boldsymbol{\beta_0 X_i}} + e^{\boldsymbol{C X_i}}e^{\boldsymbol{\beta_1 X_i}}} \\
         &= \frac{e^{\boldsymbol{\beta_1 X_i}}}{e^{\boldsymbol{\beta_0 X_i}} + e^{\boldsymbol{\beta_1 X_i}}} \\
    \end{align}
$$

So you can see that the parameter values ($\boldsymbol{\beta_0}$, $\boldsymbol{\beta_1}$) and ($\boldsymbol{\beta_0} + \boldsymbol{C}$, $\boldsymbol{\beta_1} + \boldsymbol{C}$) lead to the exact same outcome. This is problematic for fitting: how can we know which parameter set to use? Selecting one of the categories as base category circumvents this issue: we then set $\boldsymbol{\beta_0} = \boldsymbol{0}$, which leads back to the model you saw in the previous section.

Following a similar reasonning, you can expand binary logistic regression into multinomial logistic regression with $k$ categories:

$$
    \begin{align}
        P(Y_i = j) &= \frac{e^{\boldsymbol{\beta_j X_i}}}{1 + \sum_{l=1}^{k-1}e^{\boldsymbol{\beta_l X_i}}} \qquad j \in [1, k - 1] \\
        P(Y_i = 0) &= \frac{1}{1 + \sum_{l=1}^{k-1}e^{\boldsymbol{\beta_l X_i}}} \qquad\text{with }\boldsymbol{\beta_0} = \boldsymbol{0}\text{, so 0 is the base category}\\
    \end{align}
$$

This explains why you will not find any parameters for the base category when using multinomial logistic regression in practice.

## How to fit a logistic model?

Since the logistic function is non-linear in the parameters, we do not have closed-form solutions to estimate those parameters ({numref}`figure {number} <logistic_model_fit>`). Instead, we need to turn to iterative optimization methods like [gradient descent](2:beyond_least_squares:gradient_descent).

````{iframe-figure} ../../_static/part-b_logistic-model_fit.html
:name: logistic_model_fit
:width: 880px
:height: 495px

Change the slope and origin to see the impact on the model's fit. The gray area in the goodness-of-fit is where the log likelihood is undefined because the logarithm linked to diorite receives a probability of 0. The best fit corresponds to the maximum log likelihood after optimization.
````

While the sum of the squares of the residuals could be used as objective function, it is not convex with logistic regression. The (negative) log likelihood on the other hand is convex, so it used instead. Determining the likelihood follows the same principles [as with linear regression]](2:beyond_least_squares:likelihood), except that the probability density function of the normal distribution is replaced with the probability mass function of the Bernoulli distribution:

$$
    \begin{align}
        \mathcal{L}(\hat{\beta}_0, \hat{\beta}_1) &= \prod_{i=1}^n f\left(y_i|\frac{1}{1 + e^{-\left(\hat{\beta}_0 + \hat{\beta}_1 x_i\right)}}\right) \\
         &= \begin{cases}
            \prod_{i=1}^n \frac{1}{1 + e^{-\left(\hat{\beta}_0 + \hat{\beta}_1 x_i\right)}} \text{ if } y_i = 1 \\
            \prod_{i=1}^n 1 - \frac{1}{1 + e^{-\left(\hat{\beta}_0 + \hat{\beta}_1 x_i\right)}} \text{ if } y_i = 0 \\
        \end{cases} \\
         &= \prod_{i=1}^n \left(\frac{1}{1 + e^{-\left(\hat{\beta}_0 + \hat{\beta}_1 x_i\right)}}\right)^{y_i} \left(1 - \frac{1}{1 + e^{-\left(\hat{\beta}_0 + \hat{\beta}_1 x_i\right)}}\right)^{1 - y_i} \\
    \end{align}
$$

And the log likelihood becomes:

$$
    \mathcal{L}\mathcal{L}(\hat{\beta}_0, \hat{\beta}_1) = \sum_{i=1}^{n} y_i\log\left(\frac{1}{1 + e^{-\left(\hat{\beta}_0 + \hat{\beta}_1 x_i\right)}}\right) + \left(1 - y_i\right)\log\left(1 - \frac{1}{1 + e^{-\left(\hat{\beta}_0 + \hat{\beta}_1 x_i\right)}}\right)
$$

Where $\log$ stands for the natural logarithm, i.e., the logarithm to the base $e$.

When minimizing the negative log likelihood $\mathcal{N}\mathcal{L}\mathcal{L}$ with gradient descent, you need the partial derivatives with respect to each parameter for the update rule. Taking the negative of the equation above, we end up with:

$$
    \begin{align}
        \frac{\partial \mathcal{N}\mathcal{L}\mathcal{L}}{\partial \hat{\beta}_0} &= \sum_{i=1}^n (\frac{1}{1 + e^{-\left(\hat{\beta}_0 + \hat{\beta}_1 x_i\right)}} - y_i) \\
        \frac{\partial \mathcal{N}\mathcal{L}\mathcal{L}}{\partial \hat{\beta}_1} &= \sum_{i=1}^n \left(\frac{1}{1 + e^{-\left(\hat{\beta}_0 + \hat{\beta}_1 x_i\right)}} - y_i\right) x_i \\
    \end{align}
$$

## Activity: Predicting rock type from composition

Now let's have a look at applying logistic regression with Python. For that, make sure to start the interactive Python environment by clicking on {fa}`rocket` {fa}`arrow-right-long` {guilabel}`Live Code` at the top of this page (then wait until the Python interaction is ready).

First, you need to import some packages:

In [ ]:
from pathlib import Path
import numpy as np
from scipy.optimize import minimize
from scipy.special import xlogy

Then, load the data into two NumPy arrays (data from the [GEOROC Database](https://georoc.eu/georoc/new-start.asp)):

In [ ]:
rock_type = np.loadtxt(Path.cwd().parent/'../data/rock_geochem_georoc_database.csv', dtype=np.str_, delimiter=',', skiprows=1, usecols=0)
silica = np.loadtxt(Path.cwd().parent/'../data/rock_geochem_georoc_database.csv', delimiter=',', skiprows=1, usecols=1)

`silica` contains the weight percentage of silica in a rock sample, the predictor, and `rock_type` contains the type of rock, either diorite or granite, the outcome.

Let's have a look at the variable `rock_type`:

In [ ]:
rock_type

As you can see, it is made of strings, which you need to convert to numbers to use it in logistic regression. Here you only have two categories, which you can convert into probabilities of being granite.

Using NumPy, create a new array that contains 0 for diorite and 1 for granite based on `rock_type`.

In [ ]:
# Your answer here.

Now let's find estimates for the parameters of a binary logistic regression using numerical optimization. You can either re-implement gradient descent yourself or use [SciPy's function `minimize`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html), which was already imported above and requires as parameters:
  * An objective function, here the negative log likelihood, which is a Python function (in Python functions are objects, so you can use them like any other variable). That function must take as input a tuple, list, or array containing the intercept and the slope.
  * A tuple, list, or array with an initial guess for the intercept and the slope.

Rather than implementing the negative log likelihood yourself, which will lead to issues if you're not careful, use [SciPy's function `xlogy`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.xlogy.html), which was already imported above. Its documentation shows (at the bottom of the page) how to use it to implement the negative log likelihood.

In [ ]:
# Your answer here.

Predict the probability of having a granite for 67wt% silica. Which rock type is more likely?

In [ ]:
# Your answer here.

## How to analyze a logistic model?

### Parameter interpretation

The parameters in logistic regression are difficult to interpret because of the non-linear model: $\beta_0$ and $\beta_1$ are the intercept and slope of the log-odds, not of the probabilities. But the slope can be interpreted in terms of odds ratios:

$$
    OR = \frac{\text{odds}(x+1)}{\text{odds}(x)} = \frac{e^{\beta_0 + \beta_1 (x+1)}}{e^{\beta_0 + \beta_1 x_i}} = \frac{e^{\beta_0}e^{\beta_1 x}e^{\beta_1}}{e^{\beta_0}e^{\beta_1 x}} = e^{\beta_1}
$$

So when the predictor $x$ increases by 1 unit, the odds of the outcome being of category 1 are multiplied by $e^{\beta_1}$.

Since we have to define a base category that has no parameter associated to it, the parameters for the other categories end up being defined relative to that base category. This does not alter interpretation in the binary case, but it complexifies it significantly for the multinomial one. Indeed, the parameter estimates can change depending on which category is selected as base, making it difficult to perform [regression analysis like with linear regression](2:regression:interpretation).

### Pseudo R<sup>2</sup>

The [coefficient of determination](2:regression:determination) $R^2$ of linear regression is not directly transferable to logistic regression, because it is based on normally-distributed errors. Several alternatives have been proposed, none being universally accepted because they all have limitations. The most widely used is McFadden's pseudo $R^2$:

$$R_{\mathcal{L}}^2 = 1 - \frac{\mathcal{L}\mathcal{L}_{\text{full}}}{\mathcal{L}\mathcal{L}_{\text{null}}}$$

Where:
  * $\mathcal{L}\mathcal{L}_{\text{full}}$ is the log likelihood of the full model, i.e., with intercept and slope:
    $$y_i \sim \text{Bernoulli}\left(\frac{1}{1 + e^{-\left(\beta_0 + \beta_1 x_i\right)}}\right)$$

  * $\mathcal{L}\mathcal{L}_{\text{null}}$ is the log likelihood of the null model, i.e., with only the intercept:
    $$y_i \sim \text{Bernoulli}\left(\frac{1}{1 + e^{-\beta_0}}\right)$$

Adding a parameter usually increases the log likelihood compared with the more basic, less flexible null model. A big increase results in a pseudo $R^2$ getting closer to 1, mimicking the behavior of the coefficient of determination.

### Confidence intervals and significance

[Similarly to linear regression](2:uncertainty:parameters), you can get confidence intervals for the slope and intercept of the logistic regression. The standard errors follow a similar structure as those with linear regression, except that they rely on the weighted mean and variance of the predictor:

$$
    \begin{align}
        s_{\hat{\beta}_1} &= \frac{1}{\sqrt{\sum_{i=1}^n \hat{\sigma}_i^2 (x_i - \overline{x}_w)^2}} \\
        s_{\hat{\beta}_0} &= s_{\hat{\beta}_1}\sqrt{\frac{\sum_{i=1}^n\hat{\sigma}_i^2 x_i^2}{\sum_{i=1}^n\hat{\sigma}_i^2}} \\
    \end{align}
$$

With:

$$
    \begin{align}
        \hat{\sigma}_i^2 &= \hat{P}(Y_i = 1)(1 - \hat{P}(Y_i = 1)) \\
        \overline{x}_w &= \frac{\sum_{i=1}^n \hat{\sigma}_i^2 x_i}{\sum_{i=1}^n\hat{\sigma}_i^2} \\
    \end{align}
$$

Getting estimates for the 100$\gamma$% confidence intervals of the slope and the intercept is then based on a normal distribution:

$$
    \begin{align}
        \hat{\beta}_1 &\pm z_{\alpha/2} \times s_{\hat{\beta}_1} \\
        \hat{\beta}_0 &\pm z_{\alpha/2} \times s_{\hat{\beta}_0} \\
    \end{align}
$$

Where $\gamma = 1 - \alpha$. The assumption behind this formulas is that the [central limit theorem](1:clt:clt) applies. Contrary to linear regression, we do not use Student's $t$-distribution because the variance of a Bernoulli distribution ($p(1-p)$) fully depends on the mean ($p$), so it is considered known.

Testing the significance of the parameters is, again, similar to [what is done with linear regression](2:uncertainty:test_params), except that we use a $Z$-test instead of a $t$-test, for the same reason as explained in the previous paragraph.

## Generalized linear models

Linear and logistic regression can be unified as generalized linear models, which take the form:

$$
    y_i \sim \text{Distribution}\left(\text{link function}^{-1}\left(\beta_0 + \beta_1 x_i\right)\right)
$$

In the case of linear regression, its probabilistic model is:

$$y_i \sim \mathcal{N}(\beta_0 + \beta_1 x_i, \sigma_\epsilon^2)$$

So the distribution is the normal distribution and the link function is the [identity function](https://en.wikipedia.org/wiki/Identity_function).

In the case of logistic regression, its probabilistic model is:

$$y_i \sim \text{Bernoulli}\left(\frac{1}{1 + e^{-\left(\beta_0 + \beta_1 x_i\right)}}\right)$$

So the distribution is the Bernoulli distribution and the link function is the logit function.

The advantage of this more generic formulation comes from being able to define other models for other types of data, like for instance the Poisson model to predict count data:

$$y_i \sim \text{Pois}\left(e^{\beta_0 + \beta_1 x_i}\right)$$

So the distribution is the Poisson distribution and the link function is the natural logarithm function.